# Buffered Files and Prefix Queries

CSC-239 · Module 8 · Lesson 2 of 3

You can read or write a small file as one String. Now you will process text a line at a time, close buffered resources correctly, and turn dictionary lines into complete-word and prefix queries.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Read and write text lines with buffered resources and automatic cleanup.
- Build a unique dictionary and distinguish complete-word membership from a prefix match.


## Why This Matters

A Ghost player needs to know whether the current fragment is a complete dictionary word and whether any dictionary word begins with it. A dictionary file supplies the starting entries.


## Check Your Starting Point

Explain UTF-8, a Path in a new temporary directory, HashSet duplicate handling, and try-with-resources. Recall that null means no referenced object, while an empty String is an existing String with length 0.

**My explanation:**


## Concept

### Process a character stream

A **character stream** supplies or accepts decoded text characters over time. A reader provides characters from an input source. A writer sends characters to an output destination. Here the source or destination is a UTF-8 file.

Here, stream means a flow of input or output characters.

**Buffering** holds a group of data in memory so the program does not need a separate underlying I/O operation for each small request. A buffered writer may retain text briefly before sending it onward. A buffered reader may read ahead and serve later requests from its buffer.

### Write complete lines and close before reading

A **buffered text writer** collects character output and writes it efficiently. Files.newBufferedWriter opens it with the requested encoding. Like whole-file writing, its default mode creates a missing file or replaces an existing file's contents.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.BufferedWriter;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("words.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("cat");
        writer.newLine();
        writer.write("dog");
        writer.newLine();
    }
    System.out.print(Files.readString(file, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

This prints cat and dog on separate lines. write adds exactly the supplied text. newLine writes the platform's line separator. The workspace uses Linux line endings; readLine in the next example can recognize common line-ending forms.

The inner try-with-resources closes the writer before the read begins. Closing sends any remaining buffered text to the file. Do not leave the writer open and assume another reader can already see every character you gave it. A close failure is still an I/O failure and can reach the outer catch.

### Distinguish a blank line from the end

A **buffered line reader** returns each line's text without its line-ending characters. Call readLine for the next line. **End of file** means no further input remains; readLine reports this condition with null.

A blank line is different: readLine returns an empty String. That String is not null, so there may still be more lines after it.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.BufferedReader;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-read-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "cat\n\ndog", StandardCharsets.UTF_8);
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

This prints `Line: [cat]`, `Line: []`, and `Line: [dog]`. The final dog line has no trailing newline, but it still contains text and is returned. Only the next read reaches end of file.

Notice the progress step at the end of the loop: read the next line. Without that update, the condition keeps referring to the same non-null String. Use null from readLine as the end-of-file test. A blank line is data, so continue reading after it.

### Choose a dictionary loading rule

Our dictionary rule is to keep each nonempty line exactly as written and store it in a HashSet. Repeated equal words become one member. Empty lines are skipped because they are not dictionary entries for this tutorial.

We do not silently trim spaces or change case. The word cat and the word Cat remain different Strings. When using an instructor-supplied dictionary, follow its specified rules for spaces and letter case instead of assuming every file has the same format.

### Ask about the beginning of a word

A **prefix match** checks whether a String begins with a given sequence of characters. String.startsWith returns a boolean result:

```java
System.out.println("cart".startsWith("ca"));
System.out.println("cart".startsWith("ar"));
System.out.println("cart".startsWith(""));
```

This prints true, false, and true. A prefix must start at index 0. The empty prefix matches every String because it imposes no starting characters.

Complete-word membership and prefix matching answer different questions. A dictionary containing cart may report contains("ca") as false while hasPrefix("ca") is true. In our helper, hasPrefix traverses the stored words and returns true as soon as it finds a startsWith match. If the loop finds none, it returns false.

A HashSet's unspecified traversal order does not change this boolean result. We ask whether any word matches, not which word appears first. For an empty dictionary, there is no word to satisfy even an empty prefix, so our helper returns false. For a nonempty dictionary, an empty prefix returns true.

### Keep the tutorial and assignment contracts clear

The independent task defines a tutorial PrefixBook with a constructor, size, contains, and hasPrefix. Its purpose is to practice file reading and query behavior.

Your graded assignment provides FileTextReader, FileTextWriter, AbstractFileMonitor, and AbstractDictionary. Read those original declarations for their exact required methods, argument types, return types, and failure rules. The course plan names them but does not include their method declarations. PrefixBook does not replace those supplied files.


## Video Demonstration

Watch the loader distinguish blank lines from end of file, then use the set for two Boolean prefix queries. Predict the unique word count and query results before execution.

<video controls preload="metadata" width="960">
  <source src="media/02_buffered_files_and_prefix_queries/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_buffered_files_and_prefix_queries/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the buffered files and prefix queries demonstration transcript](media/02_buffered_files_and_prefix_queries/transcript.md).


## Worked Example

**Subgoal 1: build a repeatable dictionary fixture.** Write cat, cart, a blank line, dog, and repeated cat to a new file.

**Subgoal 2: load nonempty lines.** A buffered reader advances until null and a set removes repeated words.

**Subgoal 3: query prefixes.** WordChecks.hasPrefix scans for any word that starts with the requested fragment.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "cat\ncart\n\ndog\ncat\n", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix ca: " + WordChecks.hasPrefix(words, "ca"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Words: 3
Prefix ca: true
Prefix zz: false
```

The blank line is skipped and repeated cat does not increase the set size. Three words remain. At least one starts with ca, while none starts with zz. The answer does not depend on the set’s traversal order.


## Predict, Run, Trace, and Explain

### Predict the loaded words and prefixes

Before running, predict all three printed lines. The file contains `map`, an empty line, `moss`, `mud`, `mint`, and `map` again. The final line has no newline after it. Track the set size after each returned line. Explain whether the blank line ends reading, whether the final `map` is returned, and whether either line adds a new member. Name a stored word that can answer the `mi` query.

My predicted output:

Size after each of the six lines:

What the blank and final lines do:

A word that can satisfy mi:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the complete program in the Java kernel of your Workspace. Compare each printed line with your prediction before editing. Keep the original prediction and explain any revision. Account for every input line, including the blank and the final repeated word. Run the whole cell again and explain why its new directory and new set produce a fresh load.

My original prediction:

Actual output:

What matched or changed, and why:

How all six lines produce the final size:

Replay result and why it repeats:

### Trace the reads and the matching word

Trace every `readLine` result through the loader, ending with `null`. For each result, record whether the body runs, whether the set changes, and its size afterward. Explain why the next read happens even when the current line is empty. Identify where the reader closes. Then trace the queries: name a possible matching word for `mi` and explain why `return false` belongs after the prefix loop. The set does not promise a traversal order, so do not assign one in your explanation.

Read result | Body runs? | Set changes? | Size afterward

My complete trace through null:

Why every body execution advances:

Where the reader closes:

Why mi can return early and zz must finish checking:

<details>
<summary>Show answer</summary>

The output is `Words: 4`, `Prefix mi: true` and `Prefix zz: false`. The six lines leave sizes 1, 1, 2, 3, 4 and 4: skip the empty String, retain four different words and collapse repeated `map`. Final text is returned without a trailing newline; only the next read returns `null`. The word `mint` supplies the successful match. Each complete run creates its own directory, file and set, so the same input produces the same result without changing an earlier run’s file. The first read occurs before the loop. Each non-null result enters the body: nonempty text is offered to the set, and the next read occurs whether or not a word was added. The six sizes are 1, 1, 2, 3, 4 and 4. The following `null` ends the loop without entering its body. Leaving try-with-resources closes the reader before the queries print. A query for `mi` returns true when it reaches `mint`. A query for `zz` must check all words before returning false. The unknown traversal order does not change either result.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Prefix mi: true
Prefix zz: false
```

Common error: Treating a blank line as end of file. Counting a repeated word as a new member. Dropping final text because it has no following newline.

</details>


### Close the writer, then distinguish blank text from the end

Predict and run both complete programs. The first writer sends `red`, two line separators and `blue`, with no final separator. The comparison opens and closes a writer without writing text. For each, list every `readLine` value and predict the bracketed lines and `End reached:` result. After running, explain which blank line is real data, why the empty file has no line to print, and why final `blue` is returned. Locate the writer close before the reader opens, and explain how it sends any remaining buffered text onward.

My predicted and actual first output:

My predicted and actual empty-file output:

The readLine values in each case:

Why blank text, final text and end differ:

Why writer close comes before reading:


**First program:**


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


**Comparison program:**


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The writer creates three lines: `red`, an empty line and `blue`. Each `newLine()` supplies a separator; `write("blue")` adds no separator. Closing sends any text still held in the writer’s buffer before reading starts. The reader returns `"red"`, `""`, `"blue"` and then `null`, so the output includes `Line: []` and ends with `End reached: true`. In the comparison, closing the unused writer leaves an empty file. The first read returns `null`, the body runs zero times and only the end report prints. Try-with-resources closes each resource when its block ends. These outputs verify recovered lines; they do not measure the number of underlying I/O operations.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Line: [red]
Line: []
Line: [blue]
End reached: true
```

Common error: Expecting write to add a line separator automatically. Equating a blank line with an empty file. Assuming every write has already sent all buffered text before close.

**Check case 2.** The empty file supplies no line. Its first read returns null, so no Line report appears and the end report is true.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
End reached: true
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the buffered operations

The incomplete draft is shown for editing. Replace `WRITE_FACTORY`, `LINE_END`, `READ_FACTORY` and `ADVANCE` with `newBufferedWriter`, `newLine`, `newBufferedReader` and `readLine`, once each. Preserve both try-with-resources blocks and their order. Copy the completed program into the empty work cell. Predict its reports before running. Afterward explain what the first block closes, why `blue` needs no final newline to be read, and why the final loop statement must advance the reader.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.WRITE_FACTORY(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.LINE_END();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.READ_FACTORY(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.ADVANCE();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


My replacements and their purposes:

Predicted output:

Actual output:

Why writer close comes first:

Why the loop update and final text matter:

<details>
<summary>Show answer</summary>

Use `newBufferedWriter` to open the writer, `newLine` to write the first separator, `newBufferedReader` to open the reader and `readLine` to advance the loop. The second supplied separator creates the blank line. The writer block ends before the reader opens, so automatic close sends the remaining buffered text first. The final `blue` is returned without a following separator. Repeated reads eventually return `null`.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Line: [red]
Line: []
Line: [blue]
End reached: true
```

Common error: Leaving the loop update incomplete. Moving reading inside the still-open writer block. Expecting final text to disappear without a newline.

</details>


### Compare full words with prefixes

Add two prints immediately after the size report: `Has mi: ` with `words.contains("mi")`, then `Has mint: ` with `words.contains("mint")`. Keep both prefix queries. Predict all five lines, then run the complete program. Explain how a prefix can succeed while the same text is not a complete member. Test a second complete version with only the file text changed to `"mi"`, without a newline. Predict which results change. Keep the loader and helper unchanged.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Predicted and actual five lines:

Why Has mi and Prefix mi differ:

Predicted and actual one-word results:

Why mi also satisfies its own prefix:

<details>
<summary>Show answer</summary>

Complete `mi` membership is false because that whole word is absent, while `mint` membership is true. The `mi` prefix query succeeds because `mint` begins with it. With the one-word file `mi`, size becomes 1, complete `mi` membership is true and complete `mint` membership is false. The prefix `mi` remains true: a String starts with its own full text. Neither file has a word beginning with `zz`.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Has mi: " + words.contains("mi"));
    System.out.println("Has mint: " + words.contains("mint"));
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Has mi: false
Has mint: true
Prefix mi: true
Prefix zz: false
```

Common error: Turning contains into a prefix query. Assuming a prefix must be shorter than its matching word. Changing the loader while testing only query behavior.

**Additional test: `One complete word mi with no final newline`.** The final text is still read. Its full-word and prefix queries both succeed, while complete mint membership fails.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "mi", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Has mi: " + words.contains("mi"));
    System.out.println("Has mint: " + words.contains("mint"));
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Has mi: true
Has mint: false
Prefix mi: true
Prefix zz: false
```

</details>


### Repair a loader that stops at a blank line

The faulty draft should load every nonempty line, including words after a blank. Predict where it stops and all three results. Repair only the loop condition so it stops at the true end of file. Keep the inner condition that skips empty entries and the read at the end of the body. Put the full repair in the empty work cell and run it. Then test a complete version with file text `"\nmi"`. Explain why a blank first line must not hide the final word.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null && line.length() > 0) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


My predicted faulty output:

Why reading stops too soon:

My repair and actual output:

Leading-blank prediction and actual output:

Why empty entries are still excluded:

<details>
<summary>Show answer</summary>

The faulty `line.length() > 0` in the loop condition stops reading at the empty second line. Only `map` was stored, giving size 1 and false for both prefixes. Use `line != null` as the loop condition. The inner check still skips empty entries, while the read at the end advances past them. The original file then gives size 4, true for `mi` and false for `zz`. With `"\nmi"`, skip the initial empty String and retain final `mi`, giving size 1 with true and false for the two queries.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Prefix mi: true
Prefix zz: false
```

Common error: Removing the inner check and storing the empty line. Reading again only when the line was nonempty. Changing the expected result instead of repairing early termination.

**Additional test: `Blank first line followed by final mi`.** The null-only condition permits the empty String through the body, skips storing it and advances to mi. The final word is retained before the next read reports null.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "\nmi", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Prefix mi: true
Prefix zz: false
```

</details>


## Independent Practice

### Build the tutorial PrefixBook

Implement tutorial class `PrefixBook` with private `HashSet<String> words`. Its public constructor `PrefixBook(Path file) throws IOException` reads UTF-8 with a `BufferedReader`, keeps each nonempty line exactly as written, and closes the reader automatically. Provide `public int size()`, `public boolean contains(String word)` and `public boolean hasPrefix(String prefix)`. Membership checks a complete stored word; the prefix method reports whether any stored word starts with the supplied text.

Create a temporary directory with prefix `csc239-prefix-` and resolve `words.txt` inside it. Use a `BufferedWriter` to write entries `{"boat", "", "book", "bird", "boat"}`, calling `newLine()` after each entry. Close the writer before constructing the book. Print its size with `Words: `; membership for `book` and `bo` with `Has book: ` and `Has bo: `; then prefix results for `bo` and `cat` with `Prefix bo: ` and `Prefix cat: `. Include all imports and an outer `IOException` handler that prints `File problem: ` plus the message. Predict the five reports, run your program, and explain duplicates, the blank line, close order and the two kinds of query.

This is a tutorial class. Follow the instructor’s original `AbstractDictionary` declarations for the graded assignment.

Predicted five reports:

Actual reports:

Duplicate and blank-line handling:

Why writer close comes before construction:

Why Has bo and Prefix bo differ:


### Test boundaries and exact dictionary text

Test separate complete versions, each with its own new directory and file. Keep `PrefixBook` unchanged. Before each run, predict the size and every query result; afterward record actual results and explain any difference.

1. Use an empty entries array and add `Empty prefix: ` with `book.hasPrefix("")`.
2. Use `{"book", "book", "book"}` with the original five reports.
3. Use `{"boat", "", "bird", "boat", "book"}`. Write separators only between entries, so final `book` has no newline. Use an index loop and call `newLine()` only when the index is less than `entries.length - 1`. Keep the original five reports.
4. Restore the original entries and add the empty-prefix report.
5. Use `{"book", "Book", " book ", "", "book"}`. Keep the original reports, then add `Has Book: ` with `book.contains("Book")` and `Has spaced book: ` with `book.contains(" book ")`.

Explain why empty-prefix results differ for empty and nonempty dictionaries, why final `book` must be observed in case 3, and why case and surrounding spaces matter in case 5. Compare labeled results without assuming a set traversal order.

Case | Predicted reports | Actual reports | Explanation

Empty file and empty prefix:

Repeated book:

Final book without newline:

Nonempty dictionary and empty prefix:

Exact case and spaces:

How the final-line and exact-text tests reveal a loader mistake:


<details>
<summary>Show answer</summary>

The constructor retains `boat`, `book` and `bird`: it skips the empty line and stores repeated `boat` only once, giving size 3. Complete `book` membership is true; complete `bo` membership is false. Prefix `bo` succeeds because stored words begin with it; `cat` matches none. Closing the writer sends its remaining buffered text before construction starts reading. The constructor closes its reader automatically and declares `IOException` so the caller can handle I/O failure. Returning false after the prefix loop covers the case where no word matches. The original program provides the five baseline results below. In an empty dictionary, every query is false, including the empty prefix, because there is no stored word to test. Repeating `book` creates one member. Final `book` without a newline must be returned and make complete membership true; placing it only at the end exposes a dropped-final-line mistake. A nonempty dictionary has a word beginning with the empty String, so its empty-prefix result is true. In the exact-text case, `book`, `Book` and ` book ` are three different nonempty Strings. Repeated `book` adds nothing, while the truly empty line is skipped. All tests preserve the loader and query methods.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "book", "bird", "boat"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

Common error: Constructing the book while its writer is still open. Changing spaces or case despite the exact-text rule. Implementing contains as a prefix query. Returning false after the first nonmatching word.

**Additional test: Empty file, including empty prefix.** The writer creates an empty file. The first read returns null, so no query finds a stored word, including the empty-prefix query.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Empty prefix: " + book.hasPrefix(""));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 0
Has book: false
Has bo: false
Prefix bo: false
Prefix cat: false
Empty prefix: false
```

**Additional test: Three repeated book lines.** The set holds one book member. Complete book and prefix bo succeed; complete bo and prefix cat fail.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"book", "book", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

**Additional test: Unique final book without a trailing newline.** The writer adds separators only between entries. Final book is returned without a following newline. Since it is not stored earlier, losing it would change both Words and Has book.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "bird", "boat", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (int index = 0; index < entries.length; index = index + 1) {
            writer.write(entries[index]);
            if (index < entries.length - 1) {
                writer.newLine();
            }
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

**Additional test: Original nonempty dictionary with empty prefix.** Every String starts with the empty String. This dictionary contains words, so its prefix loop can return true.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "book", "bird", "boat"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Empty prefix: " + book.hasPrefix(""));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
Empty prefix: true
```

**Additional test: Preserve case and surrounding spaces.** Only the length-zero line is skipped. The three exact nonempty Strings remain different members, and both extra membership queries succeed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"book", "Book", " book ", "", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Has Book: " + book.contains("Book"));
    System.out.println("Has spaced book: " + book.contains(" book "));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
Has Book: true
Has spaced book: true
```

</details>


## Summary

Buffered I/O processes character data over time. A writer may hold pending text, so close it before reading the completed file. readLine removes line endings, returns an empty String for a blank line, and returns null at end of file. A dictionary can use a set for unique complete words and startsWith for prefix queries.

Close the answers. Explain the difference between an empty line, an empty file, an empty prefix, and a complete word.


## Reflection

A dictionary file contains repeated words, blank lines, and a final word without a newline. Specify your loader’s behavior for each case and design a prefix query that differs from a complete-word query.

**My design and explanation:**

Next, you will manage private files and compare snapshots to notice state changes.


## Supplemental Reading

- [Java 21 BufferedReader API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/BufferedReader.html) specifies readLine and end-of-file behavior.
- [Java 21 BufferedWriter API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/BufferedWriter.html) documents write, newLine, and close.
- [Java 21 Files API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) supplies buffered reader and writer factories.
- [String.startsWith](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#startsWith(java.lang.String)) defines prefix matching.
- [Java 21 HashSet API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/HashSet.html) explains duplicate membership.
